# 05 - Within-season forecasting

**Influenza Season Forecasting** - Notebook 5 of 6

**Purpose:** Fit the real models (ARIMA, Prophet) that 04's baselines are the floor for, under a
strict within-season leakage firewall, and report **skill over the matched baseline**, interval
**calibration**, and a **leak-detection audit**. Per CLAUDE.md this notebook stops for review before
any commit, and the audit is the part to review first.

## Non-negotiable evaluation rules (a violation invalidates the result)

- **Within-season firewall.** At decision week W, a model sees ONLY the held-out season's data
  through W (raw observed ILI, sw <= W). No week after W touches that season's fit. This is distinct
  from 02's retrospective target smoothing.
- **LOSO, identical 19-season set** as 04 (exclude 2008-09, 2009-10, 2020-21). The comparison is
  valid only on the same seasons.
- **Same decision weeks** as baseline C: sw 8, 12, 16. A model at W is compared to baseline C at the
  SAME W, never to a different-W or cross-sectional floor.
- **Peak read only from the forecast region** (weeks with sw > W). If a season's true peak already
  occurred by W, that season is a detection problem, not a forecast, and is reported separately as
  "peak already observed at W".

## Model configuration (fixed, not tuned for wins)

Both models are fit on the within-season raw ILI through W and forecast the remaining weeks; the peak
is read from the forecast region.

- **ARIMA** on `log1p(ILI)` (keeps forecasts positive), fixed order search `(1,1,1)->(0,1,1)->(1,0,0)`,
  first that fits. A documented, leakage-safe guard clips the back-transformed trajectory to
  `[0, max training-season peak]` so the occasional convergence blow-up cannot make the metrics a
  strawman. If no order fits, the season is recorded as a **skip with reason** (never imputed).
- **Prophet** with logistic growth, `cap = max training-season peak`, floor 0, no seasonality (the
  series is under one season long), `interval_width=0.8` for the calibration analysis, fit seeded
  (`seed=42`) so the Stan MAP estimate and its intervals are reproducible.

Both bounds use only the LOSO **training** seasons, never the held-out season's future.

## Framing (hypothesis under test, from the 04 finding)

04 found severity is forecastable and peak-week timing is near the noise floor. That is the
hypothesis here. **Models are not tuned to manufacture a timing win.** If timing fails to beat the
floor, that is the honest result and is reported as such. Honest negative results are valid
(CLAUDE.md). All interpretation is descriptive and tied to skill-over-baseline.

## Setup, data (02), and baseline reference (04)

In [ ]:
import warnings, logging, json
from statsmodels.tools.sm_exceptions import ConvergenceWarning
warnings.filterwarnings("ignore")
warnings.filterwarnings("default", category=ConvergenceWarning)   # keep ARIMA convergence failures visible (cf. peak_ambiguous)
logging.getLogger("cmdstanpy").setLevel(logging.CRITICAL)
logging.getLogger("prophet").setLevel(logging.CRITICAL)
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.arima.model import ARIMA
from prophet import Prophet

DATA_DIR = next((Path(p) for p in ["data/raw", "../data/raw"] if Path(p).exists()), Path("data/raw"))
RESULTS_DIR = DATA_DIR.parent.parent / "results"; FIG_DIR = DATA_DIR.parent.parent / "figures"
RESULTS_DIR.mkdir(exist_ok=True); FIG_DIR.mkdir(exist_ok=True)
EXCLUDED = {"2008-09", "2009-10", "2020-21"}
DECISION_WEEKS = [8, 12, 16]
SEED = 42                 # seed Prophet's Stan MAP fit for reproducibility (see calibration section)
np.random.seed(SEED)

def season_of(year, week):
    sy = year if week >= 40 else year - 1
    return f"{sy}-{str(sy + 1)[2:]}", sy
def sw(week):  return week - 39 if week >= 40 else week + 13
def sw_to_week(s): return s + 39 if s <= 13 else s - 13

In [ ]:
# rebuild weekly + season_table from 02 logic
ili = pd.read_csv(DATA_DIR / "ILINet.csv", skiprows=1, na_values=["X"])
_i = ili.apply(lambda r: season_of(int(r["YEAR"]), int(r["WEEK"])), axis=1)
ili["season"] = [x[0] for x in _i]; ili["ssy"] = [x[1] for x in _i]
ili["order"] = ili["WEEK"].apply(lambda w: w if w >= 40 else w + 100)
ili = ili.sort_values(["ssy", "order"]).reset_index(drop=True)
ili["sw"] = ili["WEEK"].apply(sw)
def _complete(g):
    sy = int(g["ssy"].iloc[0]); sp = sorted(g.loc[g["YEAR"] == sy, "WEEK"]); ep = sorted(g.loc[g["YEAR"] == sy + 1, "WEEK"])
    return bool(sp and sp[0] == 40 and ep and ep[0] == 1 and ep[-1] == 39)
weekly = ili[ili["season"].isin([s for s, g in ili.groupby("season") if _complete(g)])].copy()

rows = []
for s, g in weekly.groupby("season"):
    g = g.sort_values("order"); sm3 = g["% WEIGHTED ILI"].rolling(3, center=True).mean(); sm5 = g["% WEIGHTED ILI"].rolling(5, center=True).mean()
    rows.append(dict(season=s, ssy=int(g["ssy"].iloc[0]), peak_week=int(g.loc[sm3.idxmax(), "WEEK"]),
                     peak_ili_pct=round(float(sm3.max()), 3), peak_week_sm5=int(g.loc[sm5.idxmax(), "WEEK"])))
season_table = pd.DataFrame(rows).sort_values("ssy").reset_index(drop=True)
season_table["fragile_peak_week"] = season_table["peak_week"] != season_table["peak_week_sm5"]
season_table["sw_true"] = season_table["peak_week"].apply(sw)
EVAL = [s for s in season_table["season"] if s not in EXCLUDED]
ev = season_table[season_table["season"].isin(EVAL)].reset_index(drop=True)
assert len(EVAL) == 19 and int(ev["fragile_peak_week"].sum()) == 8

baseline04 = json.loads((RESULTS_DIR / "04_baseline_summary.json").read_text())
print("eval seasons:", len(EVAL), "| fragile:", int(ev["fragile_peak_week"].sum()))
print("04 baseline rows loaded:", [b["baseline"] for b in baseline04])

## Models and the per-(season, W) firewall runner

`run_one` is the only place a model touches data. It slices the held-out season to sw <= W, fits,
forecasts the weeks sw > W, reads the peak from that forecast region, and returns an audit record
(last observed week, the week the peak is read from, the true peak week, and the forecast/observed
status). The training-season peak max is passed in as the leakage-safe bound.

In [ ]:
def arima_forecast(y, h, cap):
    ly = np.log1p(np.asarray(y, float))
    for order in [(1, 1, 1), (0, 1, 1), (1, 0, 0)]:
        try:
            f = ARIMA(ly, order=order).fit().forecast(steps=h)
            traj = np.clip(np.expm1(np.asarray(f)), 0.0, cap)
            return traj, None, None, f"ARIMA{order}"
        except Exception:
            continue
    return None, None, None, "skip:no-order-fit"

def prophet_forecast(swv, y, h_sw, cap):
    try:
        ds0 = pd.to_datetime("2001-01-01")
        df = pd.DataFrame({"ds": ds0 + pd.to_timedelta(np.array(swv) * 7, "D"), "y": y, "cap": cap, "floor": 0.0})
        m = Prophet(growth="logistic", interval_width=0.8, weekly_seasonality=False,
                    daily_seasonality=False, yearly_seasonality=False)
        m.fit(df, seed=SEED)   # seed the Stan MAP optimizer for reproducible intervals
        fut = pd.DataFrame({"ds": ds0 + pd.to_timedelta(np.array(h_sw) * 7, "D"), "cap": cap, "floor": 0.0})
        fc = m.predict(fut)
        return fc["yhat"].values, fc["yhat_lower"].values, fc["yhat_upper"].values, "Prophet-logistic"
    except Exception as e:
        return None, None, None, f"skip:{type(e).__name__}"

def run_one(model, s, W, cap, split):
    g = weekly[weekly["season"] == s].sort_values("sw")
    obs = g[g["sw"] <= W]; h_sw = [int(x) for x in g["sw"] if x > W]
    rec = dict(model=model, season=s, W=W, last_obs_sw=int(obs["sw"].max()),
               true_sw=int(season_table.loc[season_table["season"] == s, "sw_true"].iloc[0]))
    rec["status"] = "forecast" if rec["true_sw"] > W else "peak_already_observed_at_W"
    if len(obs) < 4 or len(h_sw) < 1:
        rec.update(skip=f"too-short(obs={len(obs)},h={len(h_sw)})"); return rec
    y = obs["% WEIGHTED ILI"].values
    if model == "ARIMA":
        traj, lo, hi, tag = arima_forecast(y, len(h_sw), cap)
    else:
        traj, lo, hi, tag = prophet_forecast(obs["sw"].tolist(), y, h_sw, cap)
    if traj is None:
        rec.update(skip=tag); return rec
    k = int(np.argmax(traj))
    # Peak-week validity guard: if the forecast maximum is a flat plateau (attained at >1
    # horizon week - e.g. an ARIMA log-space blow-up clipped to the cap), argmax returns the
    # first plateau week, an extraction artifact rather than a located peak. Flag it so the
    # timing metrics can drop it. A strictly monotone rise (Prophet) attains its max once, at
    # the horizon end, and is a genuine if poor boundary forecast: not flagged.
    peak_ambiguous = bool(np.sum(traj >= traj[k] - 1e-9) > 1)
    rec.update(fit=tag, peak_read_sw=h_sw[k], pred_peak_ili=float(traj[k]),
               peak_ambiguous=peak_ambiguous,
               pi_lo=(None if lo is None else float(np.clip(lo[k], 0, cap))),
               pi_hi=(None if hi is None else float(np.clip(hi[k], 0, cap))))
    TRAJ.append(dict(model=model, season=s, W=W, split=split, h_sw=h_sw,
                     traj=[float(x) for x in traj],
                     lo=(None if lo is None else [float(x) for x in lo]),
                     hi=(None if hi is None else [float(x) for x in hi])))
    return rec

In [ ]:
TRAJ = []  # idempotency guard: reset immediately before the main forecast loop
records, skips = [], []
for model in ["ARIMA", "Prophet"]:
    for W in DECISION_WEEKS:
        for s in EVAL:
            cap = float(ev.loc[ev["season"] != s, "peak_ili_pct"].max())   # LOSO training bound
            r = run_one(model, s, W, cap, split="modeled")
            (skips if "skip" in r else records).append(r)
res = pd.DataFrame(records)
res["sw_pred"] = res["peak_read_sw"]
res = res.merge(ev[["season", "peak_ili_pct", "sw_true", "fragile_peak_week"]].rename(columns={"peak_ili_pct": "true_ili"}),
                on="season", suffixes=("", "_t"))
print("model fits:", len(res), "| skips:", len(skips))
if skips: print("SKIPS:", [(d["model"], d["season"], d["W"], d["skip"]) for d in skips])
else: print("no skips")

## Excluded seasons: labeled structural-break stress test

The three excluded seasons are scored separately with both models at every decision week. They
never enter `res`, the 19-season metrics, the skill table, or the calibration table. This is
**not a prospective forecast**: the cap and training reference use all 19 modeled seasons,
including seasons chronologically later than 2008-09 and 2009-10. The analysis characterizes
model behavior under structural breaks and reports every season separately, with no pooled mean.

Forecast trajectories are persisted through the module-level `TRAJ` sidecar. `run_one` still
returns the original audit record field for field, so `res` and `results/05_audit.json` cannot
acquire trajectory columns.

In [ ]:
SPECIAL_CASES = {
    "2008-09": "pandemic-adjacent: spring 2009 H1N1 emergence wave",
    "2009-10": "2009 H1N1 pandemic season",
    "2020-21": "near-total flu absence under COVID NPIs",
}
special_records, special_skips = [], []
special_cap = float(ev["peak_ili_pct"].max())
for model in ["ARIMA", "Prophet"]:
    for W in DECISION_WEEKS:
        for s in SPECIAL_CASES:
            r = run_one(model, s, W, special_cap, split="special_case")
            (special_skips if "skip" in r else special_records).append(r)

special_df = pd.DataFrame(special_records)
special_df["mechanism"] = special_df["season"].map(SPECIAL_CASES)
special_df["true_ili"] = special_df["season"].map(dict(zip(season_table.season, season_table.peak_ili_pct)))
special_df["true_peak_week"] = special_df["true_sw"].apply(sw_to_week)
special_df["pred_peak_week"] = special_df["peak_read_sw"].apply(sw_to_week)
special_df["peak_week_error"] = np.where(
    special_df["peak_ambiguous"], np.nan, special_df["peak_read_sw"] - special_df["true_sw"])
special_df["peak_week_abs_error"] = special_df["peak_week_error"].abs()
special_df["peak_ili_error"] = special_df["pred_peak_ili"] - special_df["true_ili"]
special_df["peak_ili_abs_error"] = special_df["peak_ili_error"].abs()
special_cols = ["season", "mechanism", "model", "W", "status", "peak_ambiguous",
                "true_peak_week", "pred_peak_week", "peak_week_error", "peak_week_abs_error",
                "true_ili", "pred_peak_ili", "peak_ili_error", "peak_ili_abs_error",
                "pi_lo", "pi_hi"]
special_df = special_df[special_cols].sort_values(["season", "model", "W"]).reset_index(drop=True)

traj_keys = [(d["model"], d["season"], d["W"], d["split"]) for d in TRAJ]
assert len(traj_keys) == len(set(traj_keys)), "duplicate trajectory sidecar key"
expected_trajectories = 2 * len(DECISION_WEEKS) * (len(EVAL) + len(SPECIAL_CASES)) - len(skips) - len(special_skips)
assert len(TRAJ) == expected_trajectories, (len(TRAJ), expected_trajectories)
(RESULTS_DIR / "05_trajectories.json").write_text(json.dumps(TRAJ, indent=2), encoding="utf-8")

special_json_records = json.loads(special_df.to_json(orient="records"))
special_payload = {
    "analysis_type": "excluded-from-training structural-break stress test, not a prospective forecast",
    "training_reference": "all 19 modeled seasons, including chronologically later seasons",
    "cap": special_cap,
    "records": special_json_records,
    "skips": special_skips,
}
(RESULTS_DIR / "05_special_cases.json").write_text(json.dumps(special_payload, indent=2), encoding="utf-8")

def special_to_md(d):
    cols = list(d.columns)
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = ["| " + " | ".join("" if pd.isna(v) else str(round(v, 3) if isinstance(v, float) else v) for v in row) + " |"
            for row in d.itertuples(index=False)]
    return "\n".join([head, sep] + body)

special_md = [
    "# 05 excluded-season structural-break stress test", "",
    "These seasons were excluded from all training and headline metrics. This is not a prospective forecast:",
    "the cap and training reference use all 19 modeled seasons, including chronologically later seasons.",
    "Errors are reported by season and model, never pooled across the three different failure mechanisms.", "",
]
table_cols = ["model", "W", "status", "peak_ambiguous", "true_peak_week", "pred_peak_week",
              "peak_week_abs_error", "true_ili", "pred_peak_ili", "peak_ili_abs_error", "pi_lo", "pi_hi"]
for s, mechanism in SPECIAL_CASES.items():
    special_md.extend([f"## {s}", "", f"Failure mechanism: {mechanism}.", "",
                       special_to_md(special_df[special_df.season == s][table_cols]), ""])
if special_skips:
    special_md.extend(["## Documented skips", "", f"`{special_skips}`", ""])
(RESULTS_DIR / "05_special_cases.md").write_text("\n".join(special_md), encoding="utf-8")
print(f"trajectory sidecar: {len(TRAJ)} unique records (expected {expected_trajectories})")
print(f"special-case fits: {len(special_df)} | skips: {len(special_skips)}")
print("saved results/05_trajectories.json and results/05_special_cases.{json,md}")

## D1 forecast overlay, Prophet at W=12

The six panels follow a rule fixed before inspecting forecasts: the mildest, median, and most
severe modeled seasons by `peak_ili_pct`, plus all three excluded seasons. Excluded panels use
distinct styling and the explicit structural-break label. Each panel shows raw observed ILI
through W, the actual continuation, the Prophet mean and 80% interval, and the true smoothed peak.

**Template interval deviation:** ARIMA produces no prediction intervals in this implementation,
so D1 is a Prophet-only figure and interval coverage is reported for Prophet only.

In [ ]:
ranked = ev.sort_values(["peak_ili_pct", "season"]).reset_index(drop=True)
modeled_panels = [
    (ranked.iloc[0]["season"], "mildest modeled season", False),
    (ranked.iloc[len(ranked) // 2]["season"], "median modeled season", False),
    (ranked.iloc[-1]["season"], "most severe modeled season", False),
]
assert len({s for s, _, _ in modeled_panels}) == 3
panels = modeled_panels + [(s, "structural-break stress test", True) for s in SPECIAL_CASES]
traj_lookup = {(d["model"], d["season"], d["W"], d["split"]): d for d in TRAJ}

fig, axes = plt.subplots(2, 3, figsize=(14, 8.8), sharex=True, sharey=True)
for ax, (s, panel_label, excluded) in zip(axes.flat, panels):
    split_name = "special_case" if excluded else "modeled"
    tr = traj_lookup[("Prophet", s, 12, split_name)]
    g = weekly[weekly.season == s].sort_values("sw")
    obs = g[g.sw <= 12]
    continuation = g[g.sw >= 12]
    ax.plot(obs.sw, obs["% WEIGHTED ILI"], color="#1f4e79", lw=2.4, label="observed through W=12")
    ax.plot(continuation.sw, continuation["% WEIGHTED ILI"], color="0.45", lw=1.8, label="actual continuation")
    ax.plot(tr["h_sw"], tr["traj"], color="#d62728", lw=2.1, label="Prophet mean")
    ax.fill_between(tr["h_sw"], tr["lo"], tr["hi"], color="#d62728", alpha=0.18, label="Prophet 80% interval")
    target = season_table[season_table.season == s].iloc[0]
    ax.scatter([target.sw_true], [target.peak_ili_pct], marker="*", s=130, color="black", zorder=5, label="true smoothed peak")
    ax.axvline(12, color="0.2", ls=":", lw=1)
    ax.set_title(f"{s}: {panel_label}\ntrue peak ILI {target.peak_ili_pct:.2f}", fontsize=10)
    if excluded:
        ax.set_facecolor("#fff1f0")
        for spine in ax.spines.values(): spine.set_color("#b22222"); spine.set_linewidth(1.5)
        ax.text(0.03, 0.95, "EXCLUDED FROM TRAINING\nstructural-break stress test", transform=ax.transAxes,
                va="top", ha="left", fontsize=8, color="#8b0000", weight="bold")
    ax.grid(alpha=0.2)
    ax.set_xticks([1, 8, 12, 16, 26, 39, 52])
for ax in axes[:, 0]: ax.set_ylabel("% WEIGHTED ILI")
for ax in axes[-1, :]: ax.set_xlabel("season week (sw)")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", bbox_to_anchor=(0.5, 0.965), ncol=5, fontsize=8)
fig.suptitle("D1: Prophet forecast overlays at decision week W=12", fontsize=15, y=0.995)
fig.text(0.5, 0.015,
         "Modeled panels are selected by the pre-specified mildest/median/most-severe rule. "
         "Excluded panels are labeled structural-break stress tests, not prospective forecasts. "
         "ARIMA produces no prediction intervals in this implementation, so D1 shows Prophet only.",
         ha="center", va="bottom", fontsize=8, color="0.3", wrap=True)
fig.tight_layout(rect=(0, 0.075, 1, 0.93))
fig.savefig(FIG_DIR / "10_D1_forecast_overlay.png", dpi=150, bbox_inches="tight")
plt.close(fig)
print("D1 modeled-season rule selected:", [(s, label) for s, label, _ in modeled_panels])
print("saved figures/10_D1_forecast_overlay.png")

## Leak-detection audit (review this first)

Firewall invariants, asserted hard:
1. `last_obs_sw <= W` for every fit (no post-W week entered any fit).
2. `peak_read_sw > W` for every fit (the predicted peak is read only from the forecast region).
3. LOSO: the held-out season is never in its own training bound (bound uses `season != s`).

Then per-season sanity lines (last observed week, the week the peak is read from, the true peak
week, status) and the forecast-vs-detection split by W.

In [ ]:
assert (res["last_obs_sw"] <= res["W"]).all(), "FIREWALL BREACH: a fit used data after W"
assert (res["peak_read_sw"] > res["W"]).all(), "AUDIT FAIL: a peak was read from the observed region"
print("Invariant 1 (last_obs_sw <= W): PASS")
print("Invariant 2 (peak_read_sw > W): PASS")
print("Invariant 3 (LOSO bound excludes held-out season): PASS by construction (season != s)\n")

aud = res.copy()
aud["last_obs_wk"] = aud["last_obs_sw"].apply(sw_to_week)
aud["peak_read_wk"] = aud["peak_read_sw"].apply(sw_to_week)
aud["true_wk"] = aud["sw_true"].apply(sw_to_week)
print("forecast vs detection split (seasons per W, either model identical):")
split = (aud[aud["model"] == "ARIMA"].groupby(["W", "status"]).size().unstack(fill_value=0))
print(split.to_string())
print("\nsample sanity lines (ARIMA, W=12):")
cols = ["season", "W", "last_obs_wk", "peak_read_wk", "true_wk", "status", "pred_peak_ili"]
print(aud[(aud.model=="ARIMA") & (aud.W==12)][cols].round(2).to_string(index=False))
aud.to_json(RESULTS_DIR / "05_audit.json", orient="records", indent=2)
print("\nsaved results/05_audit.json (full per-season audit, both models, all W)")

## Metrics under LOSO

peak_week MAE (weeks) and % within +/-1; peak_ili MAE and RMSE. Severity (`peak_ili`) is reported on
the **forecast subset** (true peak after W, the genuine forecast test) and on **all 19** for
completeness. Peak-**week** metrics are reported only on forecasts with a well-defined (non-plateau)
peak: an ARIMA blow-up clipped flat to the historical-max cap has no locatable peak week
(`peak_ambiguous`), so it is counted (`n_pw_ambiguous`) but excluded from the timing error rather than
scored on an argmax artifact. Timing is also reported excluding `fragile_peak_week` seasons.

In [ ]:
def wk_mae(d): return round(float((d["sw_pred"] - d["sw_true"]).abs().mean()), 2) if len(d) else float("nan")
def wk_w1(d): return round(float(100 * ((d["sw_pred"] - d["sw_true"]).abs() <= 1).mean()), 1) if len(d) else float("nan")
def ili_mae(d): return round(float((d["pred_peak_ili"] - d["true_ili"]).abs().mean()), 3)
def ili_rmse(d): return round(float(np.sqrt(((d["pred_peak_ili"] - d["true_ili"]) ** 2).mean())), 3)

metrics = []
for model in ["ARIMA", "Prophet"]:
    for W in DECISION_WEEKS:
        alld = res[(res.model == model) & (res.W == W)]
        fc = alld[alld.status == "forecast"]
        # peak-WEEK metrics only where the forecast peak is well-defined (non-plateau argmax);
        # ARIMA blow-ups clipped to the cap produce a flat top with no locatable peak week.
        fc_pw = fc[~fc.peak_ambiguous]
        fc_pw_f = fc_pw[~fc_pw.fragile_peak_week]
        metrics.append(dict(model=model, W=W, n_forecast=len(fc), n_detection=len(alld) - len(fc),
            n_pw_defined=len(fc_pw), n_pw_ambiguous=int(fc["peak_ambiguous"].sum()),
            pw_MAE_fc=wk_mae(fc_pw), pw_within1_fc=wk_w1(fc_pw),
            pw_within1_fc_exfrag=wk_w1(fc_pw_f), ili_MAE_fc=ili_mae(fc), ili_RMSE_fc=ili_rmse(fc),
            ili_MAE_all19=ili_mae(alld)))
metrics_df = pd.DataFrame(metrics)
print(metrics_df.to_string(index=False))

### Verify: Prophet's peak-at-boundary is a real property, not an extraction artifact

"Argmax always lands on the last week" could be a peak-extraction bug (argmax of any monotone curve
is its endpoint). It is not: for these seasons at W=12, Prophet's forecast region is monotone
increasing across the whole horizon, so the boundary argmax is genuine logistic-trend extrapolation.

In [ ]:
print("Prophet forecast-region monotonicity check (W=12):")
for s in ["2015-16", "2017-18", "2011-12"]:
    g = weekly[weekly.season == s].sort_values("sw"); W = 12
    obs = g[g.sw <= W]; h_sw = [int(x) for x in g.sw if x > W]
    cap = float(ev.loc[ev.season != s, "peak_ili_pct"].max())
    yhat, lo, hi, tag = prophet_forecast(obs.sw.tolist(), obs["% WEIGHTED ILI"].values, h_sw, cap)
    yhat = np.clip(yhat, 0, cap); d = np.diff(yhat)
    print(f"  {s}: monotone_increasing={bool((d >= -1e-9).all())} | "
          f"first(sw{h_sw[0]})={yhat[0]:.2f} -> last(sw{h_sw[-1]})={yhat[-1]:.2f} | "
          f"argmax at sw{h_sw[int(np.argmax(yhat))]} (horizon end) | smallest step={d.min():+.4f}")
print("\nConclusion: forecast region is monotone increasing, so argmax = horizon endpoint is a real")
print("property of logistic-trend extrapolation, NOT a peak-extraction artifact.")

## Skill over the matched baseline (the actual point)

Each model at W is compared to **baseline C at the same W** (within-season running max through W),
on the same forecast-subset seasons. `skill = baseline_error - model_error`; positive means the model
beats the floor, `<= 0` means it does not. Climatology is shown as a cross-sectional reference only
(the models are within-season, so C at the same W is the correct floor).

In [ ]:
def baseline_C(s, W):
    g = weekly[(weekly.season == s) & (weekly.sw <= W)]
    mx = g["% WEIGHTED ILI"].max(); mw = int(g.loc[g["% WEIGHTED ILI"] == mx, "WEEK"].iloc[0])
    return sw(mw), float(mx)

skill_rows = []
for model in ["ARIMA", "Prophet"]:
    for W in DECISION_WEEKS:
        fc = res[(res.model == model) & (res.W == W) & (res.status == "forecast")].copy()
        fc["C_sw"], fc["C_ili"] = zip(*[baseline_C(s, W) for s in fc["season"]])
        # timing skill only where the model's peak-week is well-defined (non-plateau)
        fc_pw = fc[~fc.peak_ambiguous]
        m_pw = wk_mae(fc_pw)
        c_pw = round(float((fc_pw["C_sw"] - fc_pw["sw_true"]).abs().mean()), 2) if len(fc_pw) else float("nan")
        pw_skill = round(c_pw - m_pw, 2) if len(fc_pw) else float("nan")
        m_ili = ili_mae(fc); c_ili = round(float((fc["C_ili"] - fc["true_ili"]).abs().mean()), 3)
        skill_rows.append(dict(model=model, W=W, n=len(fc), n_pw=len(fc_pw),
            pw_model=m_pw, pw_baselineC=c_pw, pw_skill=pw_skill,
            ili_model=m_ili, ili_baselineC=c_ili, ili_skill=round(c_ili - m_ili, 3)))
skill_df = pd.DataFrame(skill_rows)
print(skill_df.to_string(index=False))
print("\n(pw_skill / ili_skill > 0 => model beats baseline C at that W; <= 0 => it does not)")

## PRIMARY RESULT - Calibration: Prophet's intervals are pinned to the historical ceiling

The strongest affirmative finding in this notebook, independent of point-forecast skill. **The
mechanism is the contribution; the coverage number is only its symptom.**

**Mechanism (the cause).** Prophet's logistic-growth forecast climbs toward the training-max ceiling
(~7-7.5 `% WEIGHTED ILI`), so its 80% interval at the predicted peak sits up near that ceiling. But
most seasons peak well below it: at W=12 the median 80% interval is [7.09, 7.54] and 14 of 17 forecast
seasons fall **under** it (2 above, 1 inside), a median **2.20 ILI points** beneath its lower bound
(range 0.64 to 3.96).
The intervals are cap-pinned and almost every real peak is far underneath. This is the **same
upward-extrapolation pathology as the point forecasts**, now expressed in the uncertainty bands.

**Coverage (the symptom).** Empirical LOSO coverage (fraction of forecast-subset seasons whose true
`peak_ili_pct` falls inside the predicted **80%** interval) is **5.9-10.5% across decision weeks (1-2 of
12-19 seasons), versus the nominal 80%**: an order of magnitude too low, i.e. severely overconfident.
The per-season inside/below/above split is written to `results/05_survivorship.md`.

*Stability / reproducibility.* The per-W point estimate is granular at n~17 (one season = ~6
percentage points). W=12 in particular tips between **5.9% and 11.8%** depending on Stan optimizer
convergence, because two seasons sit within 0.04 ILI of their interval edge. The Prophet fit is seeded
(`seed=42`) so the reported numbers are reproducible; the order-of-magnitude undercoverage is robust
regardless of which way those borderline seasons fall. (The value 11.8% recorded in an earlier
unseeded run was the unlucky ~1-in-8 outcome; the modal result is 5.9%.) Quote the **range and the
mechanism**, not a single hard percentage.

In [ ]:
cal_rows = []
for W in DECISION_WEEKS:
    fc = res[(res.model == "Prophet") & (res.W == W) & (res.status == "forecast")].dropna(subset=["pi_lo", "pi_hi"])
    inside = ((fc["true_ili"] >= fc["pi_lo"]) & (fc["true_ili"] <= fc["pi_hi"]))
    cov = round(float(100 * inside.mean()), 1)
    cal_rows.append(dict(W=W, n=len(fc), nominal=80.0, empirical_coverage=cov,
                         verdict=("over-confident" if cov < 80 else "under-confident" if cov > 80 else "calibrated")))
cal_df = pd.DataFrame(cal_rows)
print(cal_df.to_string(index=False))

# mechanism diagnostic: intervals cap-pinned near the ceiling, true peaks far below
mw = res[(res.model == "Prophet") & (res.W == 12) & (res.status == "forecast")].dropna(subset=["pi_lo", "pi_hi"])
below = int((mw["true_ili"] < mw["pi_lo"]).sum())
print(f"\nmechanism @W=12: {below}/{len(mw)} forecast seasons have their true peak BELOW the interval")
print(f"  median 80% interval = [{mw['pi_lo'].median():.2f}, {mw['pi_hi'].median():.2f}] (pinned near the historical-max cap)")
print(f"  true-peak range = [{mw['true_ili'].min():.2f}, {mw['true_ili'].max():.2f}] (median gap below the interval reported in the diagnostics cell)")

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.axhline(80, color="green", ls="--", label="nominal 80%")
ax.plot(cal_df["W"], cal_df["empirical_coverage"], "o-", color="#d62728", lw=2, label="Prophet empirical")
for _, r in cal_df.iterrows(): ax.annotate(f"{r.empirical_coverage:.0f}%", (r.W, r.empirical_coverage), textcoords="offset points", xytext=(0, 8))
ax.set_xticks(DECISION_WEEKS); ax.set_xlabel("decision week W (sw)"); ax.set_ylabel("coverage of 80% PI (%)")
ax.set_ylim(0, 100); ax.set_title("Prophet 80% interval coverage on peak_ili_pct (LOSO, forecast subset)")
ax.legend(); ax.grid(alpha=0.3); fig.tight_layout()
fig.savefig(FIG_DIR / "08_prophet_calibration.png", dpi=120); plt.close(fig)
print("saved figures/08_prophet_calibration.png")

### Supplementary diagnostics (recorded, not just printed)

Two numbers quoted in the findings deck were previously computed inline and never persisted, so no
reader could check them against a committed artifact. Both are written to `results/05_survivorship.md`.

1. **Prophet interval split.** Coverage alone does not say *how* the interval misses. Split each
   forecast season into inside / below / above the 80% band.
2. **ARIMA defined-peak survivorship.** Peak-week metrics drop `peak_ambiguous` (plateau) forecasts.
   That exclusion is not neutral: check whether the retained subset is systematically easier, and
   re-score ARIMA against LOSO climatology on the *same* seasons rather than against the weak
   running-max timing rule.

In [ ]:
NL = "\n"

# --- 1. Prophet 80% interval split -------------------------------------------------
proph = res[(res.model == "Prophet") & (res.status == "forecast")].dropna(subset=["pi_lo", "pi_hi"])
split_rows = []
for W in DECISION_WEEKS:
    d = proph[proph.W == W]
    inside = (d.true_ili >= d.pi_lo) & (d.true_ili <= d.pi_hi)
    below, above = d.true_ili < d.pi_lo, d.true_ili > d.pi_hi
    gap = (d.pi_lo - d.true_ili)[below]
    split_rows.append(dict(W=W, n=len(d), inside=int(inside.sum()), below=int(below.sum()),
        above=int(above.sum()), coverage_pct=round(float(100 * inside.mean()), 1),
        pi_lo_median=round(float(d.pi_lo.median()), 2), pi_hi_median=round(float(d.pi_hi.median()), 2),
        below_gap_median=round(float(gap.median()), 2), below_gap_min=round(float(gap.min()), 2),
        below_gap_max=round(float(gap.max()), 2)))
split_df = pd.DataFrame(split_rows)

# --- 2. ARIMA defined-peak survivorship --------------------------------------------
# LOSO climatology (baseline A's rule) restricted to an arbitrary season subset: the honest
# timing floor. Baseline C (running max through W) is a weak timing rule by construction,
# because at W many seasons have not peaked, so its "prediction" is the last observed max.
sw_true_map = dict(zip(ev["season"], ev["sw_true"]))

def loso_clim_mae(seasons):
    errs = [abs(round(float(np.mean([v for k, v in sw_true_map.items() if k != s]))) - sw_true_map[s])
            for s in seasons]
    return round(float(np.mean(errs)), 2) if errs else float("nan")

skill_lut = {(r["model"], r["W"]): r for r in skill_df.to_dict("records")}
surv_rows = []
for W in DECISION_WEEKS:
    d = res[(res.model == "ARIMA") & (res.W == W) & (res.status == "forecast")]
    kept, drop = d[~d.peak_ambiguous], d[d.peak_ambiguous]
    surv_rows.append(dict(W=W, n_forecast=len(d), n_defined=len(kept), n_ambiguous=len(drop),
        arima_pw_MAE=wk_mae(kept), arima_pw_within1=wk_w1(kept),
        baselineC_pw_MAE=skill_lut[("ARIMA", W)]["pw_baselineC"],
        climatology_pw_MAE_matched=loso_clim_mae(list(kept.season)),
        kept_mean_true_ili=round(float(kept.true_ili.mean()), 2) if len(kept) else float("nan"),
        dropped_mean_true_ili=round(float(drop.true_ili.mean()), 2) if len(drop) else float("nan")))
surv_df = pd.DataFrame(surv_rows)
surv_df["edge_vs_climatology"] = (surv_df.climatology_pw_MAE_matched - surv_df.arima_pw_MAE).round(2)

print(split_df.to_string(index=False))
print()
print(surv_df.to_string(index=False))

# to_md() is redefined in the summary cell below; defined here too so this cell stands alone.
def to_md(df):
    cols = list(df.columns)
    head = "| " + " | ".join(cols) + " |"
    sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return NL.join([head, sep] + body)

_s = split_rows[1]
_v = surv_df.to_dict("records")[2]
_md = [
    "# 05 supplementary diagnostics (derived from the LOSO fits)",
    "",
    "Both tables back claims made in `slides/influenza_findings_deck.pptx`, recorded here so every",
    "slide number is checkable against a committed artifact.",
    "",
    "## 1. Prophet 80% interval split",
    "",
    "`below` means the true peak fell under the interval's lower bound (the cap-pinning pathology);",
    "`below_gap_*` is `pi_lo - true_ili` over those seasons, in `% WEIGHTED ILI` points.",
    "",
    to_md(split_df),
    "",
    f"Undercoverage is dominated by `below`, not by narrow intervals: at W=12 the median interval is",
    f"[{_s['pi_lo_median']}, {_s['pi_hi_median']}], {_s['below']} of {_s['n']} seasons fall under it,",
    f"{_s['above']} above, and only {_s['inside']} is covered.",
    "",
    "## 2. ARIMA defined-peak survivorship (peak-week timing)",
    "",
    "Peak-week metrics use non-plateau forecasts only. An ARIMA log-space blow-up clipped to the",
    "historical-max cap has no locatable peak, so it is dropped. **The dropped seasons are the severe",
    "ones**, and the retained subset is selected by the model's own success.",
    "`climatology_pw_MAE_matched` is LOSO climatology on that same subset: the honest timing floor.",
    "",
    to_md(surv_df),
    "",
    f"**Reading of W=16.** ARIMA's `pw_skill` of +3.00 against baseline C is real but misleading.",
    f"Baseline C (running max through W) is a weak *timing* rule: at W=16 many seasons have not yet",
    f"peaked, so its predicted peak week is merely the last observed maximum. Against LOSO climatology",
    f"on the same {_v['n_defined']} seasons ({_v['climatology_pw_MAE_matched']} wk), ARIMA's edge is",
    f"{_v['edge_vs_climatology']} weeks at n={_v['n_defined']}, and only {_v['arima_pw_within1']}% land",
    f"within +/-1 week, below the 36.8% floor reported in 04. The {_v['n_ambiguous']} seasons ARIMA drops",
    f"have mean peak ILI {_v['dropped_mean_true_ili']} versus {_v['kept_mean_true_ili']} for the",
    f"{_v['n_defined']} it keeps: it fails on the severe seasons and is scored on the mild ones.",
    "",
    "Conclusion: the timing result stands as reported. No model beats the timing floor in a way that",
    "survives the selection effect.",
    "",
]
(RESULTS_DIR / "05_survivorship.md").write_text(NL.join(_md), encoding="utf-8")
(RESULTS_DIR / "05_survivorship.json").write_text(json.dumps(
    {"prophet_interval_split": split_rows, "arima_defined_subset": surv_df.to_dict("records")},
    indent=2), encoding="utf-8")
print()
print("saved results/05_survivorship.md and .json")


## Leak-detection verdict

A real leak on 19 seasons would show as peak-week accuracy far above the 04 floor (every baseline was
~27-37% within +/-1). The rule: if any model/W posts `pw_within1 > 60%`, halt and treat it as a
suspected leak (audit the firewall) rather than a result.

In [ ]:
SUS = 60.0
flagged = metrics_df[(metrics_df["pw_within1_fc"] > SUS)]
print("suspicious (pw_within1_fc > %.0f%%): %s" % (SUS, flagged[["model","W","pw_within1_fc"]].to_dict("records") if len(flagged) else "NONE"))
if len(flagged) == 0:
    print("\nVERDICT: no suspected leak. Timing accuracy is at or below the 04 floor, consistent with")
    print("the firewall holding. The audit invariants (last_obs<=W, peak read from forecast region)")
    print("already passed, so the low accuracy is an honest model limitation, not a hidden win.")
else:
    print("\nVERDICT: SUSPECTED LEAK - do not report as a result until the firewall audit explains it.")

## Read: skill over baseline, and the severity/timing hypothesis

Descriptive, tied to skill-over-baseline. (Filled by the cells above.)

- **Timing.** No model beats the timing floor in a way that survives scrutiny. Prophet's logistic-trend forecast
  rises monotonically across the whole forecast region, so its argmax always lands on the horizon
  boundary (sw52 = wk39): a genuine if poor boundary forecast (0% within +/-1), not an extraction
  artifact. ARIMA usually blows up in log space and is clipped to the training-max cap, leaving a flat
  plateau with **no locatable peak week**; those cases are flagged `peak_ambiguous` and dropped from the
  timing metric (ARIMA yields a defined peak in only 9/19, 4/17, 7/12 forecasts at W = 8/12/16). On the
  seasons where it is defined it still loses to the floor at W = 8/12 (MAE ~25-31 wk). At W = 16 it posts
  `pw_skill = +3.00` against baseline C (MAE 3.00 vs 6.00), which is real but must not be quoted bare:
  the 7 retained seasons are selected by ARIMA's own success (the 5 it drops are the severe ones, mean
  peak ILI 5.75 vs 4.29), baseline C is a weak *timing* rule at W=16, and against LOSO climatology on the
  same 7 seasons (3.29 wk) the edge is only 0.29 wk at n=7 with 1 of 7 within +/-1. See
  `results/05_survivorship.md`. Consistent with the 04 finding that peak-week timing is near the noise
  floor at these lead times. No leak.
- **Severity.** The models are bounded to the historical range but still overshoot mild seasons
  (extrapolated rise), so they generally do not beat the within-season severity floor either; the
  skill column shows the sign per W. Severity remains the more tractable target (04), but these
  particular off-the-shelf models do not exploit it.
- **Calibration (the headline affirmative result).** Prophet's 80% intervals are pinned near the
  historical-max ceiling while most true peaks fall below them (median 2.20 ILI points at W=12), so
  empirical coverage is 5.9-10.5% (1-2 of 12-19 seasons) vs nominal 80%: severely overconfident, by the same
  upward-extrapolation pathology as the point forecasts. The strongest standalone finding here (see
  the primary-result section above).
- **Scope.** Capturing the epidemic turnover would need structure these models lack (a curve / mechanistic
  model); that is future work, not implemented here. No model was tuned to chase a timing win.

These are honest results against the floor, not a claim that peak forecasting is solved or impossible.

In [ ]:
summary = {"eval_seasons": len(EVAL), "decision_weeks": DECISION_WEEKS,
           "metrics": metrics_df.to_dict("records"), "skill_vs_baselineC": skill_df.to_dict("records"),
           "prophet_calibration": cal_df.to_dict("records"),
           "leak_flagged": flagged[["model","W","pw_within1_fc"]].to_dict("records"),
           "skips": [{k: d[k] for k in ("model","season","W","skip")} for d in skips]}
(RESULTS_DIR / "05_forecasting_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

def to_md(df):
    cols = list(df.columns); head = "| " + " | ".join(cols) + " |"; sep = "| " + " | ".join("---" for _ in cols) + " |"
    body = ["| " + " | ".join("" if pd.isna(v) else str(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join([head, sep] + body)

# ARIMA's +3.00 peak-week skill at W=16 is a selection artifact: timing metrics drop the plateau
# (peak_ambiguous) forecasts, and the seasons dropped are the severe ones. The skill table must
# never be emitted without the caveat that explains it. Numbers: results/05_survivorship.md.
_CAVEAT = (
    "\n> **Do not read `ARIMA / W=16 / pw_skill = +3.00` as a timing win.** Peak-week metrics are computed\n"
    "> only on the 7 of 12 forecasts where ARIMA yields a non-plateau peak, and the 5 it drops are the\n"
    "> severe seasons. Against LOSO climatology on the same 7 seasons the edge is 0.29 weeks at n=7.\n"
    "> See `05_survivorship.md`.\n")

(RESULTS_DIR / "05_forecasting_summary.md").write_text(
    "# 05 forecasting (LOSO, within-season firewall)\n\n## Metrics\n\n" + to_md(metrics_df) +
    "\n\n## Skill vs baseline C (forecast subset)\n\n" + to_md(skill_df) +
    "\n\n## Prophet 80% calibration\n\n" + to_md(cal_df) + "\n" + _CAVEAT, encoding="utf-8")
print("saved results/05_forecasting_summary.md and .json")


## Conclusion (05: ARIMA + Prophet)

Within-season ARIMA / Prophet, evaluated under a strict firewall on the same 19 seasons and decision
weeks as the baselines, do not beat the naive floor at these lead times in any way that survives the
selection effect on peak-week metrics (see `results/05_survivorship.md`): timing is at the noise floor
(no leak), and these particular models do not exploit the more tractable severity target. The
affirmative result is the **calibration finding**: Prophet's 80% intervals are cap-pinned near the
historical ceiling while real peaks fall well below, so they cover the truth only 5.9-10.5% of the time
(1-2 of 12-19 seasons) versus a nominal 80%, a clean demonstration of severe overconfidence (the same
upward-extrapolation pathology as the point forecasts) that stands on its own regardless of
point-forecast skill. Reported as honest negative point-forecast results plus that calibration
contribution. Any future model must clear the same floors under the same firewall to count.

**Next:** `06_regression_and_curve.ipynb` adds a univariate severity regression, an explanatory ridge, and a Gaussian curve fit under the same firewall.